# Milestone 1 – Reproduce: Cropped ShallowFBCSPNet on BCIC IV 2a

Reproduces the braindecode example `plot_bcic_iv_2a_moabb_cropped` with 100 epochs
for subject 3 and for all 9 subjects.

Runtime: **GPU** (Runtime → Change runtime type → T4 GPU). CPU also works but is slower.

Results and the downloaded dataset are written to Google Drive so a disconnected
session can resume (finished subjects are skipped).

In [ ]:
!nvidia-smi -L || echo "No GPU, running on CPU"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/bci_mi_decoding'
DATA_DIR = f'{DRIVE_DIR}/mne_data'
RESULTS_DIR = f'{DRIVE_DIR}/results'

In [ ]:
import os
REPO = 'github.com/Semih2341/Deep_Learning_Project.git'
# Private repo: add a GitHub token as Colab secret "GH_TOKEN" (key icon in the left bar).
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None
url = f'https://{token}@{REPO}' if token else f'https://{REPO}'
if not os.path.exists('/content/Deep_Learning_Project'):
    !git clone -q {url} /content/Deep_Learning_Project
%cd /content/Deep_Learning_Project
!pip install -q -r requirements.txt

## 1. Subject 3 (tutorial setting)

Same as the tutorial: the held-out session `1test` is used as validation split and
`EarlyStopping(patience=10, load_best=True)` restores the best epoch on it.
Target: **≈ 83.7 %**.

In [ ]:
!python src/reproduce.py --subjects 3 --mode tutorial --epochs 100 --data-dir {DATA_DIR} --out-dir {RESULTS_DIR}

## 2. All 9 subjects – tutorial setting

Target: mean close to **73.7 %** (Schirrmeister et al., 2017, ShallowConvNet).

In [ ]:
!python src/reproduce.py --subjects all --mode tutorial --epochs 100 --data-dir {DATA_DIR} --out-dir {RESULTS_DIR}

## 3. All 9 subjects – clean setting (no test-set model selection)

The tutorial picks the checkpoint using the test session. Here we train for a fixed
100 epochs without a validation split and evaluate the final model once on `1test`.

In [ ]:
!python src/reproduce.py --subjects all --mode clean --epochs 100 --data-dir {DATA_DIR} --out-dir {RESULTS_DIR}

## 4. Summary

In [ ]:
!python src/summarize.py --results-dir {RESULTS_DIR}

In [ ]:
import json, glob
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay

classes = ['feet', 'left_hand', 'right_hand', 'tongue']
for mode in ['tutorial', 'clean']:
    files = sorted(glob.glob(f'{RESULTS_DIR}/{mode}_seed*/subject_*.json'))
    if not files:
        continue
    y_true, y_pred = [], []
    for f in files:
        r = json.load(open(f))
        y_true += r['y_true']; y_pred += r['y_pred']
    disp = ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred, display_labels=classes, normalize='true', values_format='.2f')
    disp.ax_.set_title(f'{mode}: all subjects pooled')
    plt.show()